## **Ecoregion Map of South America**
-----
#### SDS 210 - Programming with Spatial Data

*Author: Isabelle Bartholet*  
*Date: 22. May 2026*

### **Research Question**
* Which biome has the most fires and are there biomes without any fires?


### **Content**

1. Load important libraries and packages 
2. Call map key via function check_map_key()
3. Fetch the fire data via function fetch_data()
4. Convert fire data to gdf
5. Import ecoregions shapefile (Downloaded from https://ecoregions.appspot.com/ -> see ReadMe.pdf)
6. Filter for polygons of South America and simplify them.
7. Perform a spatial join to combine the FIRMS fire data with the ecoregions shapefile
8. Arrange data and visualize a map using GeoJson()


### **Sources**
For detailed information see ReadMe.pdf


------
#### **Important Libraries and Map Key** 

In [1]:
import requests
import pandas as pd
import geopandas as gpd
import folium
import numpy as np

from config import MAP_KEY
from access_mapkey import check_map_key
from FetchFireData2 import fetch_data

#### **Access Map Key**

In [2]:
check_map_key(MAP_KEY) #calls function to verify API key status and remaining transactions

transaction_limit             5000
current_transactions             0
transaction_interval    10 minutes
dtype: object


{'transaction_limit': 5000,
 'current_transactions': 0,
 'transaction_interval': '10 minutes'}

#### **Import Data for the Last Five Days**

In [26]:
# fetch_data is a defintion which loads the fire data
df_fire = fetch_data(MAP_KEY)

# check length of df_fire, to see whether it worked correctly
print(f"There are {len(df_fire)} rows in this data frame.")

# quickly check dataframe by using head
df_fire.head(6)


Request successful


 Data download successful! 8091 fires found.
There are 8091 rows in this data frame.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,-4.36924,-37.89744,316.02,0.38,0.43,2026-05-18,338,N20,VIIRS,n,2.0NRT,289.88,1.77,N
1,-3.59496,-38.85813,333.97,0.45,0.47,2026-05-18,338,N20,VIIRS,n,2.0NRT,289.87,5.98,N
2,-3.59429,-38.86220,328.44,0.45,0.47,2026-05-18,338,N20,VIIRS,n,2.0NRT,289.02,5.69,N
3,-14.48688,-46.51822,338.81,0.48,0.64,2026-05-18,340,N20,VIIRS,n,2.0NRT,290.84,3.66,N
4,-14.48594,-46.52267,310.14,0.48,0.64,2026-05-18,340,N20,VIIRS,n,2.0NRT,290.06,3.66,N
5,-14.05726,-44.47939,299.95,0.36,0.57,2026-05-18,340,N20,VIIRS,n,2.0NRT,289.47,0.53,N


#### **Convert Fire Dataframe into GDF and Check for NAs**

In [4]:
# create gdf 
gdf_fire = gpd.GeoDataFrame(
    df_fire, 
    geometry=gpd.points_from_xy(
        df_fire.longitude, df_fire.latitude),
        
        crs="EPSG:4326")

# check, to see if gdf returns the same information as the df above 
gdf_fire.head(6)


# check for NAs
gdf_fire.info()


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 8091 entries, 0 to 8090
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   latitude    8091 non-null   float64 
 1   longitude   8091 non-null   float64 
 2   bright_ti4  8091 non-null   float64 
 3   scan        8091 non-null   float64 
 4   track       8091 non-null   float64 
 5   acq_date    8091 non-null   str     
 6   acq_time    8091 non-null   int64   
 7   satellite   8091 non-null   str     
 8   instrument  8091 non-null   str     
 9   confidence  8091 non-null   str     
 10  version     8091 non-null   str     
 11  bright_ti5  8091 non-null   float64 
 12  frp         8091 non-null   float64 
 13  daynight    8091 non-null   str     
 14  geometry    8091 non-null   geometry
dtypes: float64(7), geometry(1), int64(1), str(6)
memory usage: 1.1 MB


#### **Filter Data for Confidence Values**

The confidence attribute identifies the quality of every hotspot/fire pixel. It accounts for sun glint contamination during the day. Areas with low confidence (l) are associated with areas of sun glint. Nominal confidence pixels (n) are free of potential sun glint contamination. Pixels showing high confidence (h) are indicative for saturated day or nighttime pixels.  
(NASA: VIIRS I-Band 375m Active Fire Data, URL: https://www.earthdata.nasa.gov/data/instruments/viirs/viirs-i-band-375-m-active-fire-data, last access: 22.05.2026)


In [5]:
gdf_fire_reduced = gdf_fire[(gdf_fire["confidence"]!="l")].copy()
gdf_fire_reduced.head(10)

# check if no low confidence fires are included

gdf_fire_reduced["confidence"].unique()

#other method
#gdf_fire_reduced[gdf_fire_reduced["confidence"] == "l"]

<ArrowStringArray>
['n', 'h']
Length: 2, dtype: str

#### **Load Worldwide Shapefile**

1. Load worldwide shapefile 
2. Filter only polygons for South America 
3. Simplify the polygons to reduce data volume



In [6]:
## load worldwide shapefile
gdf_world_ecoregions = gpd.read_file("../data/Ecoregions2017.zip")

## filter for only polygons in  South America; column REALM == "Neotropic"
gdf_ecoregions_sa = gdf_world_ecoregions[(gdf_world_ecoregions["REALM"]=="Neotropic")].copy()

gdf_ecoregions_sa.head(6)


,OBJECTID,ECO_NAME,BIOME_NUM,BIOME_NAME,REALM,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE,geometry
21,22.0,Alto Paraná Atlantic forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Neotropic,NO01,4,439,205.740939,42.742563,Nature Imperiled,#267400,#38A700,#EE1E23,CC-BY 4.0,"MULTIPOLYGON (((-52.31779 -28.42464, -52.29635..."
22,23.0,Amazon-Orinoco-Southern Caribbean mangroves,14.0,Mangroves,Neotropic,NO14,1,611,139.824908,3.346216,Half Protected,#E600AA,#FE01C4,#257339,CC-BY 4.0,"MULTIPOLYGON (((-44.58923 -3.0014, -44.59199 -..."
36,37.0,Apure-Villavicencio dry forests,2.0,Tropical & Subtropical Dry Broadleaf Forests,Neotropic,NO02,3,520,44.540782,5.587565,Nature Could Recover,#ABE038,#CCCD65,#F9A91B,CC-BY 4.0,"MULTIPOLYGON (((-74.26596 2.95891, -74.25538 2..."
38,39.0,Araucaria moist forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Neotropic,NO01,3,440,68.503449,19.521012,Nature Could Recover,#70A800,#38A700,#F9A91B,CC-BY 4.0,"MULTIPOLYGON (((-52.22039 -29.32238, -52.23367..."
39,40.0,Araya and Paria xeric scrub,13.0,Deserts & Xeric Shrublands,Neotropic,NO13,4,597,10.258896,0.434596,Nature Imperiled,#B29841,#CC6767,#EE1E23,CC-BY 4.0,"MULTIPOLYGON (((-62.74015 10.7487, -62.75399 1..."
46,47.0,Atacama desert,13.0,Deserts & Xeric Shrublands,Neotropic,NO13,2,598,18.786639,9.200860,Nature Could Reach Half Protected,#F18650,#CC6767,#7BC141,CC-BY 4.0,"POLYGON ((-69.42981 -25.18036, -69.49561 -25.3..."


In [7]:
## check crs and geommetry
print(gdf_ecoregions_sa.geometry.head())
print(gdf_ecoregions_sa.crs)  

21    MULTIPOLYGON (((-52.31779 -28.42464, -52.29635...
22    MULTIPOLYGON (((-44.58923 -3.0014, -44.59199 -...
36    MULTIPOLYGON (((-74.26596 2.95891, -74.25538 2...
38    MULTIPOLYGON (((-52.22039 -29.32238, -52.23367...
39    MULTIPOLYGON (((-62.74015 10.7487, -62.75399 1...
Name: geometry, dtype: geometry
EPSG:4326


##### **Simplify the Polygons**

Simplify the polygons, so the map is easier to load and to work with.


In [9]:
gdf_ecoregions_sa["geometry"] = gdf_ecoregions_sa.geometry.simplify(
    tolerance = 0.05, #degree of simplification in degrees
    preserve_topology = True
).copy() 

gdf_ecoregions_sa.head(4)


,OBJECTID,ECO_NAME,BIOME_NUM,BIOME_NAME,REALM,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE,geometry
21,22.0,Alto Paraná Atlantic forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Neotropic,NO01,4,439,205.740939,42.742563,Nature Imperiled,#267400,#38A700,#EE1E23,CC-BY 4.0,"MULTIPOLYGON (((-52.31779 -28.42464, -52.19617..."
22,23.0,Amazon-Orinoco-Southern Caribbean mangroves,14.0,Mangroves,Neotropic,NO14,1,611,139.824908,3.346216,Half Protected,#E600AA,#FE01C4,#257339,CC-BY 4.0,"MULTIPOLYGON (((-44.58923 -3.0014, -44.65545 -..."
36,37.0,Apure-Villavicencio dry forests,2.0,Tropical & Subtropical Dry Broadleaf Forests,Neotropic,NO02,3,520,44.540782,5.587565,Nature Could Recover,#ABE038,#CCCD65,#F9A91B,CC-BY 4.0,"MULTIPOLYGON (((-74.35937 3.05428, -74.2162 2...."
38,39.0,Araucaria moist forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Neotropic,NO01,3,440,68.503449,19.521012,Nature Could Recover,#70A800,#38A700,#F9A91B,CC-BY 4.0,"MULTIPOLYGON (((-52.22039 -29.32238, -52.27636..."


#### **Create a Simple Ecoregion's Map**

This map is simply created to check whether the import of the shapefile has worked

In [10]:
## initialize basemap centerd on South America
# define bounding box to restrict panning outside South America
min_lon, max_lon = -81.5, -35.0
min_lat, max_lat = -65.0, 20.5


eco_region_base = folium.Map(
    max_bounds = True,
    location = [9, -64],  #9 = latitute, 9 degrees north of equator, -64 = longitude, 64 degrees west of the prime meridian
    zoom_start = 3, 
    tiles = "CartoDB Positron",
    control_scale = True,
    min_lat = min_lat,
    max_lat = max_lat,
    min_lon = min_lon,
    max_lon = max_lon
                         
)

## add choropleth map
folium.Choropleth(
    geo_data=gdf_ecoregions_sa,
    data = gdf_ecoregions_sa,
    name = "Biome Types South America",
    columns=['ECO_ID', 'BIOME_NUM'],
    key_on='feature.properties.ECO_ID', #exact path to the key inside the GeoJSON structure
    fill_color='viridis',  
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Biome Types South America'
).add_to(eco_region_base)


## add popup via an invisible GeoJSON layer
# source: https://python-visualization.github.io/folium/latest/user_guide/ui_elements/popups.html

folium.GeoJson(
    gdf_ecoregions_sa, 
    name = "Interactive Tooltips",
    # make polygons completely transparent, so they do not hide the choropleth colors
    style_function=lambda x:{"fillColor": "#ffffff00", "color":
"#ffffff00"},
popup = folium.GeoJsonPopup(
    fields =["ECO_NAME", "BIOME_NAME"],
    aliases =['Region:', 'Biome:'],
    localize = True,
    ),
).add_to(eco_region_base)


## add layer control
folium.LayerControl().add_to(eco_region_base)


eco_region_base.save("ecoregions_sa_popup.html")




#### **Combine Fires with Biome-Map**

This is done to create a more sophisticated biome map where the fires are included.

1. Check the crs of both gdfs -> should be EPSG: 4326
2. Perform spatial join according to some conditions
3. Count how many fires are per biome
4. Include it in the final map

In [11]:
 ## check CRS of Fire GDF and Biome GDF
print(gdf_fire_reduced.crs)
print(gdf_ecoregions_sa.crs)


EPSG:4326
EPSG:4326


In [12]:
## perform join according to conditions 

# keep Fire geometry and attach Biome geometry
# where the Fire is "within" the biome polygon

fires_within_biomes = gpd.sjoin(gdf_fire_reduced, gdf_ecoregions_sa,
                                how = "inner", predicate = "within")

fires_within_biomes.head(3)

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE
0,-4.36924,-37.89744,316.02,0.38,0.43,2026-05-18,338,N20,VIIRS,n,...,NO02,4,525,113.696455,60.155581,Nature Imperiled,#81B50A,#CCCD65,#EE1E23,CC-BY 4.0
1,-3.59496,-38.85813,333.97,0.45,0.47,2026-05-18,338,N20,VIIRS,n,...,NO02,4,525,113.696455,60.155581,Nature Imperiled,#81B50A,#CCCD65,#EE1E23,CC-BY 4.0
2,-3.59429,-38.86220,328.44,0.45,0.47,2026-05-18,338,N20,VIIRS,n,...,NO02,4,525,113.696455,60.155581,Nature Imperiled,#81B50A,#CCCD65,#EE1E23,CC-BY 4.0


In [20]:
## count the number of fires per biome
# define bounding box to restrict panning outside South America
fires_per_biome = (
    fires_within_biomes.groupby("BIOME_NAME").size().reset_index(name="FIRE_COUNT")   #
)

fires_per_biome = fires_per_biome.sort_values("FIRE_COUNT", ascending=False)

display(fires_per_biome)

#.size = counts rows per group





,BIOME_NAME,FIRE_COUNT
8,"Tropical & Subtropical Grasslands, Savannas & ...",3172
9,Tropical & Subtropical Moist Broadleaf Forests,1529
7,Tropical & Subtropical Dry Broadleaf Forests,690
6,"Temperate Grasslands, Savannas & Shrublands",584
0,Deserts & Xeric Shrublands,540
3,"Mediterranean Forests, Woodlands & Scrub",473
5,Temperate Broadleaf & Mixed Forests,372
1,Flooded Grasslands & Savannas,362
4,Montane Grasslands & Shrublands,34
2,Mangroves,21


#### **Merge Fire Counts with Ecoregions**

This merge is necessary, so the fire counts can be included in the popup of the map and the ecoregions are still visible.

In [14]:
## merge fires_per_biomes with gdf_ecorgeions_sa

gdf_ecoregions_sa = gdf_ecoregions_sa.merge(
    fires_per_biome, 
    on = "BIOME_NAME",
    how = "left"
)

missing_count = gdf_ecoregions_sa["FIRE_COUNT"].isna().sum()
print(f"There are {missing_count} missing values in FIRE_COUNT gdf_ecosystems_sa.")


display(fires_within_biomes.head(3))

There are 8 missing values in FIRE_COUNT gdf_ecosystems_sa.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE
0,-4.36924,-37.89744,316.02,0.38,0.43,2026-05-18,338,N20,VIIRS,n,...,NO02,4,525,113.696455,60.155581,Nature Imperiled,#81B50A,#CCCD65,#EE1E23,CC-BY 4.0
1,-3.59496,-38.85813,333.97,0.45,0.47,2026-05-18,338,N20,VIIRS,n,...,NO02,4,525,113.696455,60.155581,Nature Imperiled,#81B50A,#CCCD65,#EE1E23,CC-BY 4.0
2,-3.59429,-38.86220,328.44,0.45,0.47,2026-05-18,338,N20,VIIRS,n,...,NO02,4,525,113.696455,60.155581,Nature Imperiled,#81B50A,#CCCD65,#EE1E23,CC-BY 4.0


#### **Handling NA-rows**

From above, one knows in the *FIRE_COUNT* column of gdf_ecosystems_sa are eight missing values. This could be because in one biome there are no fires. 

In [21]:
# Check how many rows are in the dataset gdf_ecoregions_sa
print(gdf_ecoregions_sa["BIOME_NAME"].unique())

# Check how many rows are in the dataset gdf_ecoregions_sa
print(fires_per_biome["BIOME_NAME"].unique())


<ArrowStringArray>
[          'Tropical & Subtropical Moist Broadleaf Forests',
                                                'Mangroves',
             'Tropical & Subtropical Dry Broadleaf Forests',
                               'Deserts & Xeric Shrublands',
 'Tropical & Subtropical Grasslands, Savannas & Shrublands',
                'Tropical & Subtropical Coniferous Forests',
                          'Montane Grasslands & Shrublands',
                 'Mediterranean Forests, Woodlands & Scrub',
                            'Flooded Grasslands & Savannas',
              'Temperate Grasslands, Savannas & Shrublands',
                      'Temperate Broadleaf & Mixed Forests']
Length: 11, dtype: str
<ArrowStringArray>
['Tropical & Subtropical Grasslands, Savannas & Shrublands',
           'Tropical & Subtropical Moist Broadleaf Forests',
             'Tropical & Subtropical Dry Broadleaf Forests',
              'Temperate Grasslands, Savannas & Shrublands',
                        

The lenght of the *BIOME_NAME* column is 11 for the gdf_ecoregions_sa. For the fires_per_biome the length is 10. Therefore, in one biome there are no fires. When comparing the two ArrowStringArray outputs from above, the biome *Tropical & Subtropical Coniferous Forests* are missing in the lower ArrowStringArraw. Hence, in this biome there were no fires detected with this satellite. As eight values are missing, it could be that this biome comprises of eight polygons. The following lines of code will determine this. 

In [22]:
## figure out which biome is missing in the fires_per_biome 

counts = (gdf_ecoregions_sa.groupby("BIOME_NAME").size().reset_index(name= "# Polygons"))

counts = counts.sort_values("# Polygons", ascending = False)
print(counts)


                                           BIOME_NAME  # Polygons
10     Tropical & Subtropical Moist Broadleaf Forests          81
8        Tropical & Subtropical Dry Broadleaf Forests          32
0                          Deserts & Xeric Shrublands          14
9   Tropical & Subtropical Grasslands, Savannas & ...          11
4                     Montane Grasslands & Shrublands           9
7           Tropical & Subtropical Coniferous Forests           8
1                       Flooded Grasslands & Savannas           8
2                                           Mangroves           7
5                 Temperate Broadleaf & Mixed Forests           4
6         Temperate Grasslands, Savannas & Shrublands           4
3            Mediterranean Forests, Woodlands & Scrub           1


As in the output above visible, the number of polygons for the biome *Tropical & Subtropical Coniferous Forests* is 8. Thus, this supports the assumption that no fires where detected whithin this biome. Therefore, 0 will be assigned to the NA values.

In [23]:
## assign 0 to every NA value in the FIRE_COUNT column

# make a save copy first
gdf_ecoregions_sa_filled = gdf_ecoregions_sa.copy()


# fill NA values with 0
gdf_ecoregions_sa_filled["FIRE_COUNT"] = gdf_ecoregions_sa_filled["FIRE_COUNT"].fillna(0)

## check, whether the filling worked
print(gdf_ecoregions_sa_filled["FIRE_COUNT"].isna().sum())

0


#### **Create Map With Fire Count Included in Popups**



Check the names of the *BIOME_NAME* column (for creating a nice legend)

In [18]:
print(gdf_ecoregions_sa_filled["BIOME_NAME"].unique())

<ArrowStringArray>
[          'Tropical & Subtropical Moist Broadleaf Forests',
                                                'Mangroves',
             'Tropical & Subtropical Dry Broadleaf Forests',
                               'Deserts & Xeric Shrublands',
 'Tropical & Subtropical Grasslands, Savannas & Shrublands',
                'Tropical & Subtropical Coniferous Forests',
                          'Montane Grasslands & Shrublands',
                 'Mediterranean Forests, Woodlands & Scrub',
                            'Flooded Grasslands & Savannas',
              'Temperate Grasslands, Savannas & Shrublands',
                      'Temperate Broadleaf & Mixed Forests']
Length: 11, dtype: str


In [ ]:
## create map coloured according to biomes and with fire data in the popup

# initialize basemap centerd on South America
# define bounding box to restrict panning outside South America
min_lon, max_lon = -81.5, -35.0
min_lat, max_lat = -65.0, 20.5


biome_basemap = folium.Map(
    max_bounds = True,
    location = [9, -64],  #9 = latitute, 9 degrees north of equator, -64 = longitude, 64 degrees west of the prime meridian
    zoom_start = 3, 
    tiles = "CartoDB Positron",
    control_scale = True,
    min_lat = min_lat,
    max_lat = max_lat,
    min_lon = min_lon,
    max_lon = max_lon
                         
)

## create colormap dictionary for legend
# source for colours: https://waldyrious.net/viridis-palette-generator/

biome_colors = {
    "Tropical & Subtropical Moist Broadleaf Forests": "#fef38c",
    "Mangroves": "#bddf26",
    "Tropical & Subtropical Dry Broadleaf Forests": "#7ad151",
    "Deserts & Xeric Shrublands": "#44bf70",
    "Tropical & Subtropical Grasslands, Savannas & Shrublands": "#22a884",
    "Tropical & Subtropical Coniferous Forests": "#21918c",
    "Montane Grasslands & Shrublands": "#2a788e",
    "Mediterranean Forests, Woodlands & Scrub": "#355f8d",
    "Flooded Grasslands & Savannas": "#414487",
    "Temperate Grasslands, Savannas & Shrublands": "#482475",
    "Temperate Broadleaf & Mixed Forest": "#440154"
}



## add ecoregion polygons colored by biome type; 
folium.GeoJson(
    gdf_ecoregions_sa_filled,
    name="Biome Types South America",
    style_function=lambda x: {
        "fillColor": biome_colors.get(x["properties"]["BIOME_NAME"], "#cccccc"),
        "color": "white", #border line
        "weight": 0.5, # how thick the border line is
        "fillOpacity": 1,
    }, 
    #lambda x will be called for every polygon

    popup=folium.GeoJsonPopup(
        fields=["ECO_NAME", "BIOME_NAME", "FIRE_COUNT"],
        aliases=["Region:", "Biome:", "Total #Fires per Biome:"],
        localize=True,
    ),
).add_to(biome_basemap)



## html legend

legend_html = """
<div style = "position: fixed;
bottom: 50px; 
right: 50px;
width: 180px; 
height: 350px;
background-color: lightgrey;
z-index:9999;
font-size:10px;
padding: 11px;
border: 2px solid grey;
">

<b>Biome Name</b><br>
<i style="background:#fef38c;width:20px;height:10px;float:left;margin-right:8px;"></i> Tropical & Subtropical Moist Broadleaf Forests<br>

<i style="background:#bddf26;width:20px;height:10px;float:left;margin-right:8px;"></i> Mangroves<br>

<i style="background:#7ad151;width:20px;height:10px;float:left;margin-right:8px;"></i> Tropical & Subtropical Dry Broadleaf Forests<br>

<i style="background:#44bf70;width:20px;height:10px;float:left;margin-right:8px;"></i> Deserts & Xeric Shrublands<br>

<i style="background:#22a884;width:20px;height:10px;float:left;margin-right:8px;"></i> Tropical & Subtropical Grasslands, Savannas & Shrublands<br>

<i style="background:#21918c;width:20px;height:10px;float:left;margin-right:8px;"></i> Tropical & Subtropical Coniferous Forests<br>

<i style="background:#2a788e;width:20px;height:10px;float:left;margin-right:8px;"></i> Montane Grasslands & Shrublands<br>

<i style="background:#355f8d;width:20px;height:10px;float:left;margin-right:8px;"></i> Mediterranean Forests, Woodlands & Scrub<br>

<i style="background:#414487;width:20px;height:10px;float:left;margin-right:8px;"></i> Flooded Grasslands & Savannas<br>

<i style="background:#482475;width:20px;height:10px;float:left;margin-right:8px;"></i> Temperate Grasslands, Savannas & Shrublands<br>

<i style="background:#440154;width:20px;height:10px;float:left;margin-right:8px;"></i> Temperate Broadleaf & Mixed Forest

</div>
"""

# add html legend to map
biome_basemap.get_root().html.add_child(folium.Element(legend_html))


## add layer control
folium.LayerControl().add_to(biome_basemap)


biome_basemap.save("biome_map_popup_withfires.html")

biome_basemap



#### **Results**

3. Which biome has the most fires and are there biomes without any fires?

To answer research question three the biome map combined with fire data is studied. From the code, which was written, the biome with the most fires is Tropical & Subtropical Grasslands, Savannas & Shrublands. Having 2960 detected fires over the past five days. The biome Tropical & Subtropical Coniferous Forests does not have any fires in the analysed time range. When considering the biome map and comparing it to the observations of the time animated heatmap before, the biome Tropical & Subtropical Grasslands, Savannas & Shrublands is mostly found where the hotspots on the heatmap are visible, namely, also in eastern Brazil, Venezuela and Paraguay. The two findings, therefore, align. However, as the duration of this study is extremely limited, it cannot be concluded that the biome Tropical & Subtropical Grass-lands, Savannas & Shrublands has generally the most fires or that the biome Tropical & Subtropical Coniferous Forests does never experience any fires. 